# 10 — Final Evaluation on the Test Split

**Goal:** report the performance of the winning (model, strategy) combination — selected in notebook 09 — on the 16 held-out test notes.

**Validity of this estimate:** the test split played no role in model selection or prompt design, and all prompt examples (few-shot pool and cluster medoids) come from the population, outside the annotated sample. This evaluation is run **once**, only for the winner — running it for all combinations and reporting the best would reintroduce selection bias.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from clinical_notes_extraction.utils.evaluation import (
    evaluate_note, evaluate_split, locate_spans,
)

In [ ]:
# Constants local to this notebook
RESULTS_DIR = Path("results/test")
GROUND_TRUTH_FILE = Path("data/annotations/ground_truth.json")
SPLIT_FILE = Path("data/splits/test.csv")
BEST_COMBO_FILE = Path("results/best_combination.json")
TEST_METRICS_FILE = Path("results/test_metrics.csv")

In [ ]:
with open(BEST_COMBO_FILE) as f:
    best_combo = json.load(f)
print("Evaluating:", json.dumps(best_combo, indent=2))

## Load ground truth and predictions for the test notes

In [ ]:
with open(GROUND_TRUTH_FILE) as f:
    ground_truth_raw = json.load(f)

notes = pd.read_csv(SPLIT_FILE).set_index("note_id")

ground_truth = {}
for note_id in notes.index:
    entities = ground_truth_raw[str(note_id)]
    if entities and "start" not in entities[0]:
        entities = locate_spans(notes.loc[note_id, "text"], [e["span"] for e in entities])
        unverified = [e["span"] for e in entities if not e["verified"]]
        assert not unverified, f"Gold spans not found verbatim in note {note_id}: {unverified}"
    ground_truth[str(note_id)] = entities

pred_dir = RESULTS_DIR / best_combo["model"].replace(":", "_") / best_combo["strategy"]
predictions = {}
for pred_file in sorted(pred_dir.glob("*.json")):
    with open(pred_file) as f:
        pred = json.load(f)
    predictions[str(pred["note_id"])] = pred["medications"]

assert set(predictions) == set(ground_truth), "Prediction / ground-truth note_id mismatch"
print(f"Loaded {len(predictions)} test predictions from {pred_dir}")

## Metrics on the test split

Per-note counts are shown alongside the aggregate: with 16 notes, note-level variability is part of the story and belongs in the thesis, not just a single number.

In [ ]:
per_note = pd.DataFrame([
    {"note_id": note_id,
     **{f"exact_{k}": v for k, v in evaluate_note(preds, ground_truth[note_id], "exact").items()},
     **{f"relaxed_{k}": v for k, v in evaluate_note(preds, ground_truth[note_id], "relaxed").items()}}
    for note_id, preds in predictions.items()
])
per_note

In [ ]:
test_metrics = evaluate_split(predictions, ground_truth)
test_df = pd.DataFrame([{"model": best_combo["model"],
                         "strategy": best_combo["strategy"], **test_metrics}])
test_df.to_csv(TEST_METRICS_FILE, index=False)
print(f"Saved: {TEST_METRICS_FILE}")
test_df.round(3)

## Generalisation check: train/val vs test

A large drop from train/val to test suggests the selection phase overfit to the 16 train/val notes (a real possibility with a small sample); a small gap supports the claim that the winning combination generalises.

In [ ]:
comparison = pd.DataFrame({
    "train_val": {"relaxed_f1": best_combo["train_val_relaxed_f1"],
                  "exact_f1": best_combo["train_val_exact_f1"]},
    "test": {"relaxed_f1": round(test_metrics["relaxed_f1"], 4),
             "exact_f1": round(test_metrics["exact_f1"], 4)},
})
comparison["gap"] = (comparison["train_val"] - comparison["test"]).round(4)
comparison